In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA version (PyTorch built with):", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Torch: 2.5.0+cu124
CUDA version (PyTorch built with): 12.4
CUDA available: True
GPU: NVIDIA GeForce RTX 3060


In [2]:
from tqdm import tqdm
import torch
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import io,glob,os,PIL
from PIL import Image
import transformers
from transformers import AutoModel, AutoProcessor
import matplotlib.pyplot as plt

D:\Anaconda\envs\all_in_one\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from accelerate import Accelerator
accelerator = Accelerator(mixed_precision="fp16")  # ✅ enables fp16 training on your 3060

In [4]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.verbose = False  # hide detailed logs

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
class Translation_Dataset(Dataset):
    def __init__(self,src_en, src_ur):
        with open(src_en,'r',encoding='utf-8') as en:
            self.en_lines = en.readlines()

        with open(src_ur,'r',encoding='utf-8') as ur:
            self.ur_lines = ur.readlines()
        assert len(self.en_lines) == len(self.ur_lines),"mismatch in num of lines"
    
    def __len__(self):
        return len(self.ur_lines)

    def __getitem__(self,index):
        return self.en_lines[index].strip(), self.ur_lines[index].strip()

In [7]:
# ---------- Safety + memory-friendly utilities ----------
import os, time, copy, gc, torch

save_dir = "./checkpoints_LoRa_mimic"
os.makedirs(save_dir, exist_ok=True)

def _model_state_to_cpu(model):
    sd = model.state_dict()
    cpu_sd = {}
    for k, v in sd.items():
        if isinstance(v, torch.Tensor):
            cpu_sd[k] = v.detach().cpu()
        else:
            cpu_sd[k] = v
    return cpu_sd

def _optimizer_state_to_cpu(opt_state_dict):
    cpu_state = {"state": {}, "param_groups": copy.deepcopy(opt_state_dict.get("param_groups", []))}
    for k, v in opt_state_dict.get("state", {}).items():
        new_v = {}
        for sk, sv in v.items():
            try:
                if isinstance(sv, torch.Tensor):
                    new_v[sk] = sv.detach().cpu()
                else:
                    new_v[sk] = sv
            except Exception:
                new_v[sk] = sv
        cpu_state["state"][k] = new_v
    return cpu_state

In [8]:
class DualEncoder(nn.Module):
    def __init__(self,model_name = "google/siglip2-base-patch16-224"):
        super().__init__()
        self.en_encoder = AutoModel.from_pretrained(model_name).text_model
        for params in self.en_encoder.parameters():
            params.requires_grad = False

        self.ur_encoder = AutoModel.from_pretrained(model_name).text_model
        self.proj = nn.Identity()

    def forward(self,en_inputs,ur_inputs):
        en_outputs = self.en_encoder(**en_inputs)
        en_cls = en_outputs[1]
        
        ur_outputs = self.ur_encoder(**ur_inputs)
        ur_cls = ur_outputs[1]

        return en_cls,ur_cls


model_name = "google/siglip2-base-patch16-224"
model = DualEncoder(model_name=model_name).to(device)
processor = AutoProcessor.from_pretrained(model_name)

from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, PeftModel
import torch

# --- 1. freeze base model weights (optional but recommended) ---
for p in model.ur_encoder.parameters():
    p.requires_grad = False

# --- 2. create LoRA config ---
lora_config = LoraConfig(
    r=64,                      # LoRA rank
    lora_alpha=16,            # scaling
    target_modules=[
        # common projector names across transformer implementations.
        # include multiple candidate names so it works for many backbones.
        "q_proj", "k_proj", "v_proj", "o_proj",
        "query", "key", "value", "dense", "proj",
    ],
    lora_dropout=0.2,
    bias="none"               # can be "all", "none" or "lora_only"
    # task_type can be left default for adapters applied to encoder models
)

# --- 3. wrap the ur_encoder with LoRA ---
model.ur_encoder = PeftModel.from_pretrained(model.ur_encoder, "./checkpoints_LoRa_mimic/lora_adapter_step23692_20251006-040149", is_trainable=True)

# model.ur_encoder = get_peft_model(model.ur_encoder, lora_config)

# Optional: sanity-check how many params are trainable
trainable_params = sum(p.numel() for p in model.ur_encoder.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.ur_encoder.parameters())
print(f"Trainable params (LoRA) = {trainable_params}, total = {total_params}")


optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=0.01
)

# option A — save the peft adapter using peft utilities (recommended)
adapter_dir = "lora_adapter"
model.ur_encoder.save_pretrained(adapter_dir)

# # option B — save only the adapter state dict (more portable)
# peft_state = get_peft_model_state_dict(model.ur_encoder)
# torch.save(peft_state, "lora_adapter_state_dict.pt")


def collate_fn(batch, max_len=64):
    en_texts, ur_texts = [], []

    for en, ur in batch:
        # tokenize separately to check length
        en_tokens = processor.tokenizer(en)["input_ids"]
        ur_tokens = processor.tokenizer(ur)["input_ids"]

        if len(en_tokens) <= max_len and len(ur_tokens) <= max_len:
            en_texts.append(en)
            ur_texts.append(ur)

    if len(en_texts) == 0:
        return None, None  # skip this batch entirely

    en_inputs = processor(text=en_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    ur_inputs = processor(text=ur_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_len)

    return en_inputs, ur_inputs

en_path = "./Dataset/EN_UR_Parallel_Dataset/EN_UR_Parallel_Dataset/English.best_for_urdu_clean_complete.en"
ur_path = "./Dataset/EN_UR_Parallel_Dataset/EN_UR_Parallel_Dataset/Urdu.best_for_urdu_clean_complete.ur"


en_path = "./Dataset/EN_UR_Parallel_Dataset/EN_UR_Parallel_Dataset/English.en"
ur_path = "./Dataset/EN_UR_Parallel_Dataset/EN_UR_Parallel_Dataset/Urdu.ur"

en_path = "./Dataset/conceptual_captions/cc.en"
ur_path = "./Dataset/conceptual_captions/cc.ur"

full_dataset = Translation_Dataset(en_path, ur_path)
batch_size = 256

# 80/10/10 split
train_size = int(0.7 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Trainable params (LoRA) = 884736, total = 283188480


In [10]:
model

DualEncoder(
  (en_encoder): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(256000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): PytorchGELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_

In [11]:
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, collate_fn=collate_fn)

model, optimizer, train_loader, val_loader, test_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader, test_loader
)
model = torch.compile(model, backend="aot_eager")  # no inductor, stable on Windows

# tracking
best_val_loss = float("inf")
num_epochs = 10
step = 0
import random
from tqdm import tqdm


def evaluate(model, loader, desc="Evaluating", max_batches=None):
    model.eval()
    total_loss, count = 0.0, 0

    with torch.no_grad():
        for i, (en_inputs, ur_inputs) in enumerate(loader):
            if en_inputs is None:  
                continue

            en_inputs = {k: v.to(device) for k, v in en_inputs.items()}
            ur_inputs = {k: v.to(device) for k, v in ur_inputs.items()}

            en_emb, ur_emb = model(en_inputs, ur_inputs)
            en_emb = F.normalize(en_emb, dim=-1)
            ur_emb = F.normalize(ur_emb, dim=-1)

            logits = en_emb @ ur_emb.T
            labels = torch.arange(logits.size(0), device=device)

            loss = F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)
            total_loss += loss.item()
            count += 1

            if max_batches and count >= max_batches:
                break

    avg_loss = total_loss / max(1, count)
    print(f"{desc} | avg_loss={avg_loss:.4f}")
    return avg_loss


import os
import time
import copy
import gc
import torch

save_dir = "./checkpoints_LoRa_mimic"
os.makedirs(save_dir, exist_ok=True)

def _to_cpu_tensor(obj):
    return obj.detach().cpu() if isinstance(obj, torch.Tensor) else obj

def _save_peft_adapter_only(model, path_base, step, timestamp):
    """
    Try to save LoRA/PEFT adapter for `model` (or model.ur_encoder).
    Returns True if adapter saved, False otherwise.
    """
    try:
        from peft import get_peft_model_state_dict
    except Exception:
        return False

    candidates = []
    if hasattr(model, "ur_encoder"):
        candidates.append(model.ur_encoder)
    candidates.append(model)

    for cand in candidates:
        try:
            peft_state = get_peft_model_state_dict(cand)
            if peft_state:
                # ensure CPU tensors
                cpu_peft = {k: _to_cpu_tensor(v) if isinstance(v, torch.Tensor) else v for k, v in peft_state.items()}
                lora_path = path_base + ".lora.pt"
                torch.save(cpu_peft, lora_path)
                print(f"✅ Saved LoRA state dict: {lora_path}")

                # try save_pretrained (more convenient to reload)
                try:
                    adapter_folder = os.path.join(save_dir, f"lora_adapter_step{step}_{timestamp}")
                    if hasattr(cand, "save_pretrained"):
                        cand.save_pretrained(adapter_folder)
                        print(f"✅ Saved LoRA adapter folder: {adapter_folder}")
                except Exception:
                    pass

                return True
        except Exception:
            continue
    return False

def save_checkpoint(model, optimizer, epoch, step, metrics=None, best=False, tag="latest", save_full=False):
    """
    Save checkpoint.
      - By default (save_full=False) this saves only:
          * a small metadata file
          * the LoRA/PEFT adapter weights if detected (tiny)
      - If save_full=True it will also attempt a CPU-safe full checkpoint (may take longer).
    model should be the unwrapped nn.Module (accelerator.unwrap_model(model)).
    """
    os.makedirs(save_dir, exist_ok=True)
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    metrics = dict(metrics) if metrics is not None else {}

    # try short suffix from metrics
    loss_suffix = ""
    try:
        if "val_loss" in metrics:
            loss_suffix = f"_val{float(metrics['val_loss']):.4f}"
        elif "train_loss" in metrics:
            loss_suffix = f"_train{float(metrics['train_loss']):.4f}"
        elif len(metrics) == 1:
            k, v = next(iter(metrics.items()))
            loss_suffix = f"_{k}{float(v):.4f}"
    except Exception:
        loss_suffix = ""

    fname = f"best_step{step}{loss_suffix}_{timestamp}.pt" if best else f"{tag}_epoch{epoch}_step{step}{loss_suffix}_{timestamp}.pt"
    path_base = os.path.join(save_dir, fname)

    # --- metadata (always small) ---
    meta = {"epoch": epoch, "step": step, "metrics": metrics, "timestamp": timestamp}
    meta_path = path_base + ".meta.pt"
    torch.save(meta, meta_path)
    print(f"✅ Saved metadata: {meta_path}")

    # --- try saving adapter only (preferred) ---
    adapter_saved = _save_peft_adapter_only(model, path_base, step, timestamp)
    if not adapter_saved:
        print("ℹ️  No LoRA/PEFT adapter detected or `peft` unavailable — adapter save skipped.")

    # --- optional full checkpoint (CPU-safe) if requested explicitly ---
    if save_full:
        def _model_state_to_cpu(m):
            sd = m.state_dict()
            out = {}
            for k, v in sd.items():
                out[k] = _to_cpu_tensor(v) if isinstance(v, torch.Tensor) else v
            return out

        def _optimizer_state_to_cpu(opt):
            if opt is None:
                return None
            cpu_state = {"state": {}, "param_groups": copy.deepcopy(opt.state_dict().get("param_groups", []))}
            for k, v in opt.state_dict().get("state", {}).items():
                new_v = {}
                for sk, sv in v.items():
                    try:
                        new_v[sk] = _to_cpu_tensor(sv) if isinstance(sv, torch.Tensor) else sv
                    except Exception:
                        new_v[sk] = sv
                cpu_state["state"][k] = new_v
            return cpu_state

        try:
            model_cpu = _model_state_to_cpu(model)
        except Exception as e:
            print(f"⚠️ Failed to gather model CPU state: {e}")
            model_cpu = None

        try:
            opt_cpu = _optimizer_state_to_cpu(optimizer)
        except Exception as e:
            print(f"⚠️ Failed to gather optimizer CPU state: {e}")
            opt_cpu = None

        full_state = {
            "epoch": epoch,
            "step": step,
            "model_state": model_cpu,
            "optimizer_state": opt_cpu,
            "metrics": metrics,
        }
        full_path = path_base + ".full.pt"
        torch.save(full_state, full_path)
        print(f"✅ Saved CPU-safe full checkpoint: {full_path}")

    # GC to reduce memory pressure
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

    # summary print
    if metrics:
        metrics_str = ", ".join(f"{k}={v:.6g}" if isinstance(v, (int, float)) else f"{k}={v}" for k, v in metrics.items())
        print(f"(metrics: {metrics_str})")

# Ensure best_val_loss exists
try:
    best_val_loss
except NameError:
    best_val_loss = float("inf")

In [ ]:
# --- Training Loop ---
for epoch in range(num_epochs):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    num_batches = len(train_loader)
    mid_point = num_batches // 2

    for batch_idx, (en_inputs, ur_inputs) in enumerate(loop):
        if en_inputs is None:
            continue

        en_inputs = {k: v.to(accelerator.device) for k, v in en_inputs.items()}
        ur_inputs = {k: v.to(accelerator.device) for k, v in ur_inputs.items()}


        en_emb, ur_emb = model(en_inputs, ur_inputs)
        en_emb = F.normalize(en_emb, dim=-1)
        ur_emb = F.normalize(ur_emb, dim=-1)
        
        # compute matrices
        logits_eu = en_emb @ ur_emb.T
        logits_ee = en_emb @ en_emb.T
        
        # mimic loss (as above)
        tau_pred = 0.1
        tau_target = 0.1
        lambda_mimic = 0.6
        lambda_ce = 0.4
        
        with torch.no_grad():
            target_probs = torch.softmax(logits_ee / tau_target, dim=1)
        
        pred_log_probs = torch.log_softmax(logits_eu / tau_pred, dim=1)
        mimic_loss = F.kl_div(pred_log_probs, target_probs, reduction='batchmean')
        
        if lambda_ce > 0:
            labels = torch.arange(logits_eu.size(0), device=logits_eu.device)
            ce_loss = F.cross_entropy(logits_eu, labels) + F.cross_entropy(logits_eu.T, labels)
        else:
            ce_loss = torch.tensor(0.0, device=logits_eu.device)

        loss = lambda_mimic * mimic_loss + lambda_ce * ce_loss
        
        loss_value = float(loss.item())   # ✅ define here


        optimizer.zero_grad()
        accelerator.backward(loss)
        optimizer.step()

        step += 1
        loop.set_postfix(loss=f"{loss.item():.4f}")
        # --- Mid-epoch evaluation (adapter-only save) ---
        if batch_idx == mid_point:
            save_checkpoint(
                accelerator.unwrap_model(model),    # unwrapped module for PEFT detection
                optimizer,
                epoch,
                step,
                metrics={"train_loss": loss_value},
                best=False,
                tag="mid",
                # save_full=False  # default: adapter-only save
            )

            val_loss = evaluate(model, val_loader, desc="Mid-Epoch Eval (full)")
            val_loss = float(val_loss.item()) if isinstance(val_loss, torch.Tensor) else float(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                # if you want to keep a CPU-safe full checkpoint for the best mid model, set save_full=True
                save_checkpoint(
                    accelerator.unwrap_model(model),
                    optimizer,
                    epoch,
                    step,
                    metrics={"val_loss": val_loss},
                    best=True,
                    tag="mid_best",
                    save_full=False  # still adapter-only by default
                )
            else:
                save_checkpoint(
                    accelerator.unwrap_model(model),
                    optimizer,
                    epoch,
                    step,
                    metrics={"val_loss": val_loss},
                    best=False,
                    tag="mid_val",
                    save_full=False
                )

    # --- End-of-epoch evaluation (adapter-only by default) ---
    val_loss = evaluate(model, val_loader, desc="End-Epoch Eval (full)")
    val_loss = float(val_loss.item()) if isinstance(val_loss, torch.Tensor) else float(val_loss)

    print(f"Epoch {epoch+1}: Val Loss = {val_loss:.4f}")

    save_checkpoint(
        accelerator.unwrap_model(model),
        optimizer,
        epoch,
        step,
        metrics={"val_loss": val_loss},
        best=False,
        tag="end",
        save_full=False  # choose True if you want the full CPU checkpoint here
    )

Epoch 1/10:   0%|                                                                            | 0/15794 [00:00<?, ?it/s]D:\Anaconda\envs\all_in_one\lib\site-packages\torch\autograd\graph.py:825: UserWarning: cuDNN SDPA backward got grad_output.strides() != output.strides(), attempting to materialize a grad_output with matching strides... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\MHA.cpp:676.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch 1/10:   0%|                                                    | 1/15794 [00:16<71:58:25, 16.41s/it, loss=0.1035]D:\Anaconda\envs\all_in_one\lib\site-packages\torch\autograd\graph.py:825: UserWarning: cuDNN SDPA backward got grad_output.strides() != output.strides(), attempting to materialize a grad_output with matching strides... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch

✅ Saved metadata: ./checkpoints_LoRa_mimic\mid_epoch0_step7898_train0.0604_20251006-081638.pt.meta.pt
✅ Saved LoRA state dict: ./checkpoints_LoRa_mimic\mid_epoch0_step7898_train0.0604_20251006-081638.pt.lora.pt
✅ Saved LoRA adapter folder: ./checkpoints_LoRa_mimic\lora_adapter_step7898_20251006-081638
(metrics: train_loss=0.060352)
Mid-Epoch Eval (full) | avg_loss=10.8831
✅ Saved metadata: ./checkpoints_LoRa_mimic\best_step7898_val10.8831_20251006-083047.pt.meta.pt
✅ Saved LoRA state dict: ./checkpoints_LoRa_mimic\best_step7898_val10.8831_20251006-083047.pt.lora.pt
✅ Saved LoRA adapter folder: ./checkpoints_LoRa_mimic\lora_adapter_step7898_20251006-083047


Epoch 1/10:  50%|██████████████████████▌                      | 7898/15794 [1:39:55<560:48:28, 255.69s/it, loss=0.0604]

(metrics: val_loss=10.8831)


D:\Anaconda\envs\all_in_one\lib\site-packages\torch\autograd\graph.py:825: UserWarning: cuDNN SDPA backward got grad_output.strides() != output.strides(), attempting to materialize a grad_output with matching strides... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\MHA.cpp:676.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch 1/10:  50%|███████████████████████                       | 7901/15794 [1:40:18<200:05:24, 91.26s/it, loss=0.0558]D:\Anaconda\envs\all_in_one\lib\site-packages\torch\autograd\graph.py:825: UserWarning: cuDNN SDPA backward got grad_output.strides() != output.strides(), attempting to materialize a grad_output with matching strides... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\MHA.cpp:676.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engin

...

Epoch 2/10:  50%|████████████████████████                        | 7897/15794 [1:21:24<1:26:14,  1.53it/s, loss=0.0771]

✅ Saved metadata: ./checkpoints_LoRa_mimic\mid_epoch1_step23692_train0.0771_20251006-112415.pt.meta.pt
✅ Saved LoRA state dict: ./checkpoints_LoRa_mimic\mid_epoch1_step23692_train0.0771_20251006-112415.pt.lora.pt
✅ Saved LoRA adapter folder: ./checkpoints_LoRa_mimic\lora_adapter_step23692_20251006-112415
(metrics: train_loss=0.0771085)
Mid-Epoch Eval (full) | avg_loss=10.8614
✅ Saved metadata: ./checkpoints_LoRa_mimic\best_step23692_val10.8614_20251006-113743.pt.meta.pt
✅ Saved LoRA state dict: ./checkpoints_LoRa_mimic\best_step23692_val10.8614_20251006-113743.pt.lora.pt
✅ Saved LoRA adapter folder: ./checkpoints_LoRa_mimic\lora_adapter_step23692_20251006-113743


Epoch 2/10:  50%|██████████████████████▌                      | 7898/15794 [1:34:53<533:43:17, 243.34s/it, loss=0.0771]

(metrics: val_loss=10.8614)


Epoch 2/10:  96%|██████████████████████████████████████████████▊  | 15095/15794 [2:46:02<06:53,  1.69it/s, loss=0.0967]